# 4.4 LMCache: Prefix KV Reuse Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.4_lmcache/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.4_lmcache/lab.ipynb)

**Goal**: Measure real GPU speedup from reusing prefix KV cache across requests sharing the same system prompt.

**Requirements**: GPU with >= 16GB VRAM (T4, A10G, L4), ~14GB for Mistral-7B in fp16.

In [ ]:
# ── INSTALL (run this cell once, then skip on subsequent runs) ──────────────────
# Using subprocess so you can run this cell independently without affecting other cells.
import subprocess, sys

def pip_install(*packages, extra_args=None):
    """Install packages via pip. extra_args: list of additional flags like ['--no-build-isolation']"""
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(packages)
    if extra_args:
        cmd += extra_args
    subprocess.check_call(cmd)

# Core dependencies
pip_install('transformers', 'accelerate')

# ────────────────────────────────────────────────────────────────────────────────


In [ ]:
# Cell 2: Imports and device setup
import time  # perf_counter for high-resolution wall-clock timing
# Execute operationimport torch  # GPU tensor operations and CUDA synchronization
# Execute operationimport matplotlib.pyplot as plt  # plotting benchmark charts
# Execute operationimport numpy as np  # numerical operations (mean, array manipulation)
# Execute operationfrom transformers import AutoTokenizer, AutoModelForCausalLM  # HF model/tokenizer loading

# Assert GPU is available -- this lab requires CUDA for meaningful KV cache benchmarks
assert torch.cuda.is_available(), "GPU required for this lab"

# Retrieve GPU name for reporting in results summary
device_name = torch.cuda.get_device_name(0)
# Get total VRAM in GB to verify we have enough for Mistral-7B fp16 (~14GB)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
# Display formatted resultprint(f"Device: {device_name} ({vram_gb:.1f} GB VRAM)")

In [ ]:
# Cell 3: Load Mistral-7B in fp16 (non-gated, no auth needed)
import os
# Disable HuggingFace telemetry to avoid OAuth popups on restricted environments
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# Mistral-7B-v0.1 chosen because it's non-gated (no auth), has GQA (8 KV heads),
# and is architecturally identical to Llama-style models used in production
MODEL_ID = "mistralai/Mistral-7B-v0.1"

# Load tokenizer -- converts text to token IDs for model input
tokenizer_mistral = AutoTokenizer.from_pretrained(MODEL_ID, use_auth_token=False)

# Load model in fp16 to fit in 16GB VRAM; device_map="auto" places layers across available GPUs
model_mistral = AutoModelForCausalLM.from_pretrained(
    # Process this step    MODEL_ID,
    # Process this step    torch_dtype=torch.float16,  # Half precision: 7B params * 2 bytes = ~14GB
    device_map="auto",  # Automatically distribute layers across available devices
    use_auth_token=False,  # No authentication needed for this public model
)
# Set to eval mode -- disables dropout, ensures deterministic KV cache behavior
model_mistral.eval()
print(f"Model loaded: {MODEL_ID} (fp16)")

In [ ]:
# Cell 4: Define shared system prompt and unique user queries
# This simulates a real serving scenario: many users share the same system prompt
# but send different questions. The system prompt KV cache can be computed ONCE
# and reused across all requests, avoiding redundant prefill computation.

# A realistic system prompt (~80 tokens) that would be shared across all users
SYSTEM_PROMPT = (
    # Process this step    "You are an expert assistant specializing in distributed systems, "
    # Process this step    "cloud infrastructure, and machine learning operations. Provide "
    # Process this step    "detailed technical answers with concrete examples. Always consider "
    # Process this step    "scalability, reliability, and cost efficiency in your responses. "
    # Process this step    "When discussing architectures, mention specific technologies and "
    # Process this step    "their tradeoffs. Focus on production-grade solutions."
# Process this step)

# 10 distinct user queries -- each shares the system prompt prefix
# but appends a unique question (simulates 10 different users)
USER_QUERIES = [
    "How does consistent hashing work?",
    "Explain the CAP theorem.",
    "What is vector clock synchronization?",
    "Describe Raft consensus protocol.",
    "How does LSM-tree storage work?",
    "What are bloom filters used for?",
    "Explain circuit breaker pattern.",
    "How does gRPC differ from REST?",
    "What is eventual consistency?",
    "Describe leader election algorithms.",
]

# Tokenize system prompt to measure its length in tokens
# This length determines how much computation we save by caching
system_tokens = tokenizer_mistral.encode(SYSTEM_PROMPT, return_tensors="pt").to("cuda")
# .shape[1] gives sequence length (shape is [batch=1, seq_len])
system_prefix_len = system_tokens.shape[1]
print(f"System prompt length: {system_prefix_len} tokens")

In [ ]:
# Cell 5: Benchmark WITHOUT prefix caching (cold prefill each time)
# Baseline: every request re-computes attention for the ENTIRE input (system + query).
# This is what happens without LMCache -- O(n^2) attention on the full sequence each time.

# Store TTFT (time to first token) for each request in milliseconds
ttft_no_cache_list = []

# Iterate over all 10 user queries
for query_idx, user_query in enumerate(USER_QUERIES):
    # Concatenate system prompt + user query (full input re-processed every time)
    full_prompt = SYSTEM_PROMPT + " " + user_query
    # Tokenize the full concatenated prompt and move to GPU
    input_ids_full = tokenizer_mistral.encode(full_prompt, return_tensors="pt").to("cuda")
    
    # cuda.synchronize() forces all pending GPU ops to finish before timing starts
    # Without this, we'd measure kernel launch time, not actual computation time
    torch.cuda.synchronize()
    
    # perf_counter gives nanosecond-resolution wall clock (best for benchmarking)
    start_time = time.perf_counter()
    with torch.no_grad():  # Disable gradient tracking -- inference only, saves memory
        # Generate exactly 1 token -- this measures TTFT (prefill + first decode step)
        output_no_cache = model_mistral.generate(
            input_ids_full,
            max_new_tokens=1,  # Only 1 token: isolates prefill latency from decode
            do_sample=False,  # Greedy decoding for deterministic results
        )
    # synchronize again to ensure the generate() kernel has fully completed on GPU
    # GPU ops are async -- without this, perf_counter would return before GPU finishes
    torch.cuda.synchronize()
    # Calculate elapsed wall-clock time for this request
    elapsed_no_cache = time.perf_counter() - start_time
    
    # Convert seconds to milliseconds and store for later comparison
    ttft_no_cache_list.append(elapsed_no_cache * 1000)
    print(f"  Request {query_idx+1:2d}: TTFT = {elapsed_no_cache*1000:.1f} ms (no cache)")

# Compute average TTFT across all 10 requests as the baseline metric
mean_ttft_no_cache = np.mean(ttft_no_cache_list)
print(f"\nMean TTFT without caching: {mean_ttft_no_cache:.1f} ms")

In [ ]:
# Cell 6: Benchmark WITH prefix KV reuse (manual past_key_values)
# LMCache approach: compute KV cache for system prompt ONCE, then each request
# only processes the unique user query tokens. Saves O(prefix_len^2) attention.

# Step 1: Pre-compute KV cache for the shared system prompt (done once)
with torch.no_grad():  # No gradients needed -- pure inference
    # Forward pass on system prompt tokens with use_cache=True
    # This computes and returns the Key/Value tensors for all transformer layers
    prefix_output = model_mistral(system_tokens, use_cache=True)
    # past_key_values is a tuple of (K, V) pairs, one per transformer layer
    # Each K/V has shape [batch, num_kv_heads, seq_len, head_dim]
    cached_kv = prefix_output.past_key_values

# Report what was cached -- confirms the prefix was processed
print(f"Prefix KV cached: {system_prefix_len} tokens")
# Show tensor shapes to illustrate GQA structure (8 KV heads for Mistral-7B)
print(f"KV shape per layer: K={cached_kv[0][0].shape}, V={cached_kv[0][1].shape}")

# Step 2: For each request, only process the user query tokens (reusing prefix KV)
# Store TTFT for cached requests in milliseconds
ttft_with_cache_list = []

# Iterate over same 10 queries but now with KV reuse
for query_idx, user_query in enumerate(USER_QUERIES):
    # Tokenize ONLY the user query (not the system prompt -- that's already cached)
    # Leading space maintains tokenization consistency with concatenated version
    query_tokens = tokenizer_mistral.encode(" " + user_query, return_tensors="pt").to("cuda")
    
    # Synchronize before timing to drain any pending GPU operations
    torch.cuda.synchronize()
    # Start high-resolution timer
    start_cached = time.perf_counter()
    with torch.no_grad():  # Inference mode -- no gradient computation
        # Key difference: pass past_key_values=cached_kv
        # The model skips computing attention over the prefix tokens entirely
        # Only the query_tokens go through full attention computation
        output_cached = model_mistral.generate(
            query_tokens,
            past_key_values=cached_kv,  # Reuse pre-computed system prompt KV
            max_new_tokens=1,  # Measure TTFT only (1 decode step)
            do_sample=False,  # Greedy for reproducibility
        )
    # Ensure GPU kernel completes before stopping timer
    torch.cuda.synchronize()
    # Calculate elapsed time for this cached request
    elapsed_cached = time.perf_counter() - start_cached
    
    # Store in ms for comparison with baseline
    ttft_with_cache_list.append(elapsed_cached * 1000)
    print(f"  Request {query_idx+1:2d}: TTFT = {elapsed_cached*1000:.1f} ms (with KV reuse)")

# Compute mean TTFT with caching for comparison
mean_ttft_with_cache = np.mean(ttft_with_cache_list)
# Speedup = how many times faster cached is vs cold (>1 means caching helps)
speedup_ratio = mean_ttft_no_cache / mean_ttft_with_cache
print(f"\nMean TTFT with prefix reuse: {mean_ttft_with_cache:.1f} ms")
print(f"Speedup: {speedup_ratio:.2f}x")

In [ ]:
# Cell 7: Sweep prefix lengths to show speedup scales with prefix size
# Hypothesis: longer shared prefixes yield greater speedup because more
# attention computation (O(n^2) in prefix length) is avoided per request.

# Test prefix lengths from 128 to 2048 tokens
PREFIX_LENGTHS = [128, 256, 512, 1024, 2048]
# Run 5 requests per prefix length for statistical stability
NUM_TRIALS = 5

# Create a long token sequence by repeating the system prompt
# We'll slice it to different lengths to test various prefix sizes
long_text = (SYSTEM_PROMPT + " ") * 50  # Repeat 50x to exceed 2048 tokens
# Tokenize the entire long text and move to GPU
long_tokens = tokenizer_mistral.encode(long_text, return_tensors="pt").to("cuda")

# A short suffix simulating a user query appended after the prefix
suffix_text = "Summarize the key point."
# Tokenize suffix separately -- this is what gets processed on each cached request
suffix_tokens = tokenizer_mistral.encode(suffix_text, return_tensors="pt").to("cuda")

# Lists to collect results for each prefix length
sweep_no_cache_means = []  # Mean TTFT without caching at each prefix length
sweep_cached_means = []  # Mean TTFT with caching at each prefix length
sweep_speedups = []  # Speedup ratio at each prefix length

# Test each prefix length
for prefix_len in PREFIX_LENGTHS:
    # Safety check: skip if we don't have enough tokens in our long sequence
    if long_tokens.shape[1] < prefix_len:
        print(f"  Skipping {prefix_len} (not enough tokens)")
        continue
    
    # Slice the first prefix_len tokens as our prefix
    prefix_slice = long_tokens[:, :prefix_len]
    # Full input = prefix + suffix (what cold prefill must process entirely)
    full_input = torch.cat([prefix_slice, suffix_tokens], dim=1)
    
    # --- Measure WITHOUT cache (cold prefill baseline) ---
    times_cold = []  # Store individual trial times
    for trial_i in range(NUM_TRIALS):
        # Synchronize to ensure clean timing start
        torch.cuda.synchronize()
        t0_cold = time.perf_counter()  # Start timer
        with torch.no_grad():  # No gradients for inference
            # Process full input (prefix + suffix) from scratch each time
            model_mistral.generate(full_input, max_new_tokens=1, do_sample=False)
        # Wait for GPU to finish before recording end time
        torch.cuda.synchronize()
        # Record elapsed time in milliseconds
        times_cold.append((time.perf_counter() - t0_cold) * 1000)
    
    # --- Pre-compute prefix KV cache for this prefix length ---
    with torch.no_grad():
        # Forward pass on prefix to build KV cache tensors
        prefix_fwd = model_mistral(prefix_slice, use_cache=True)
        # Extract KV pairs for all layers -- reused in warm trials below
        kv_for_sweep = prefix_fwd.past_key_values
    
    # --- Measure WITH cache (warm, only suffix processed) ---
    times_warm = []  # Store individual trial times
    for trial_i in range(NUM_TRIALS):
        # Synchronize before timing
        torch.cuda.synchronize()
        t0_warm = time.perf_counter()  # Start timer
        with torch.no_grad():
            # Only process suffix tokens; prefix KV comes from cache
            model_mistral.generate(
                suffix_tokens, past_key_values=kv_for_sweep,
                max_new_tokens=1, do_sample=False
            )
        # Wait for GPU completion before stopping timer
        torch.cuda.synchronize()
        # Record elapsed time in milliseconds
        times_warm.append((time.perf_counter() - t0_warm) * 1000)
    
    # Compute mean across trials for statistical reliability
    mean_cold = np.mean(times_cold)
    mean_warm = np.mean(times_warm)
    # Speedup ratio: how many times faster cached path is
    ratio = mean_cold / mean_warm
    
    # Append to result lists for plotting
    sweep_no_cache_means.append(mean_cold)
    sweep_cached_means.append(mean_warm)
    sweep_speedups.append(ratio)
    
    print(f"  Prefix {prefix_len:4d} tokens: cold={mean_cold:.1f}ms, warm={mean_warm:.1f}ms, speedup={ratio:.2f}x")

In [ ]:
# Cell 8: Plot TTFT comparison bar chart (system prompt experiment)
# Visualizes the per-request speedup from KV reuse vs cold prefill

# Create a single subplot figure, 10x5 inches for readability
fig_bar, ax_bar = plt.subplots(1, 1, figsize=(10, 5))

# x positions for each of the 10 requests
x_positions = np.arange(len(USER_QUERIES))
# Width of each bar (two bars per position: cold and warm)
bar_width = 0.35

# Red bars: cold prefill TTFT (no caching -- full recomputation)
bars_cold = ax_bar.bar(
    # Process this step    x_positions - bar_width/2, ttft_no_cache_list,  # Offset left
    # Process this step    bar_width, label="No Cache (full prefill)", color="#ef4444", alpha=0.8
# Process this step)
# Green bars: warm TTFT (with KV reuse -- only query tokens processed)
bars_warm = ax_bar.bar(
    # Process this step    x_positions + bar_width/2, ttft_with_cache_list,  # Offset right
    # Process this step    bar_width, label="With KV Reuse", color="#22c55e", alpha=0.8
# Process this step)

# Label axes and title
ax_bar.set_xlabel("Request #")
ax_bar.set_ylabel("TTFT (ms)")
ax_bar.set_title(f"TTFT: Cold Prefill vs Prefix KV Reuse ({system_prefix_len}-token prefix)")
# Set x-axis tick labels as R1, R2, ... R10
ax_bar.set_xticks(x_positions)
ax_bar.set_xticklabels([f"R{i+1}" for i in range(len(USER_QUERIES))])
# Add legend to distinguish bar colors
ax_bar.legend()
# Light horizontal grid for easier value reading
ax_bar.grid(axis="y", alpha=0.3)

# Annotate the overall speedup in top-right corner with a box
ax_bar.text(
    0.98, 0.95, f"Mean speedup: {speedup_ratio:.2f}x",
    transform=ax_bar.transAxes, ha="right", va="top",  # Position relative to axes
    fontsize=12, fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="#dbeafe", alpha=0.8)  # Blue info box
)

# Adjust layout to prevent label clipping
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Plot prefix length sweep -- demonstrates speedup scales with prefix size
# Key insight: longer prefixes = more saved computation = higher speedup

# Create 2 side-by-side subplots: absolute times (left) and speedup factor (right)
fig_sweep, (ax_times, ax_speedup) = plt.subplots(1, 2, figsize=(12, 5))

# Left plot: absolute TTFT in ms at each prefix length
# Red line with circles: cold prefill (grows with prefix length due to O(n^2) attention)
ax_times.plot(PREFIX_LENGTHS[:len(sweep_no_cache_means)], sweep_no_cache_means,
              # Process this step              "o-", color="#ef4444", linewidth=2, markersize=8, label="No Cache")
# Green line with squares: warm (stays flat since only suffix is processed)
ax_times.plot(PREFIX_LENGTHS[:len(sweep_cached_means)], sweep_cached_means,
              # Process this step              "s-", color="#22c55e", linewidth=2, markersize=8, label="With KV Reuse")
ax_times.set_xlabel("Prefix Length (tokens)")
ax_times.set_ylabel("TTFT (ms)")
ax_times.set_title("TTFT vs Prefix Length")
ax_times.legend()
ax_times.grid(alpha=0.3)

# Right plot: speedup factor (cold/warm) at each prefix length
# Bar chart showing how speedup increases with longer prefixes
ax_speedup.bar(range(len(sweep_speedups)), sweep_speedups,
               # Process this step               color="#3b82f6", alpha=0.8)
# Label each bar with the prefix length
ax_speedup.set_xticks(range(len(sweep_speedups)))
# Iterate over each item in the collectionax_speedup.set_xticklabels([str(p) for p in PREFIX_LENGTHS[:len(sweep_speedups)]])
ax_speedup.set_xlabel("Prefix Length (tokens)")
ax_speedup.set_ylabel("Speedup (x)")
ax_speedup.set_title("Speedup from KV Reuse vs Prefix Length")
# Dashed line at 1.0x to mark the break-even point
ax_speedup.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
ax_speedup.grid(axis="y", alpha=0.3)

# Prevent subplot overlap
plt.tight_layout()
# Render the chartplt.show()

In [ ]:
# Cell 10: Summary statistics -- print all key results in one place
# This cell provides a quick reference of the entire experiment's findings

# Process this stepprint("=" * 60)
# Process this stepprint("EXPERIMENT SUMMARY")
# Process this stepprint("=" * 60)
# Report which model and hardware were used for reproducibility
print(f"Model: {MODEL_ID} (fp16)")
# Display formatted resultprint(f"GPU: {device_name}")

# System prompt experiment results
print(f"\n--- System Prompt Experiment ---")
print(f"Prefix length: {system_prefix_len} tokens")  # How many tokens were cached
print(f"Mean TTFT (cold):  {mean_ttft_no_cache:.1f} ms")  # Baseline without caching
print(f"Mean TTFT (warm):  {mean_ttft_with_cache:.1f} ms")  # With KV reuse
print(f"Speedup: {speedup_ratio:.2f}x")  # How much faster caching makes it

# Prefix length sweep results -- shows scaling behavior
print(f"\n--- Prefix Length Sweep ---")
# Print speedup at each tested prefix length
for i, plen in enumerate(PREFIX_LENGTHS[:len(sweep_speedups)]):
    print(f"  {plen:4d} tokens -> {sweep_speedups[i]:.2f}x speedup")

# Key takeaway for the reader
print(f"\nKey insight: Speedup grows with prefix length because more")
print(f"computation is skipped when reusing cached KV states.")